# MedSymmFlow Synthetic Augmentation for PneumoniaMNIST Classification

**Final project notebook**

This notebook evaluates whether chest X-ray images generated by **MedSymmFlow** improve a downstream pneumonia classifier. The study includes full-data experiments, a low-data setting, image-quality analysis, threshold analysis, and a matched real-image oversampling control.

### Main research question
Can class-conditioned synthetic chest X-ray images improve the performance of a ResNet-18 classifier trained on PneumoniaMNIST?

### Key findings
- Small or moderate synthetic additions improved some full-data metrics, but the effect was not monotonic.
- Adding 176 synthetic Normal images produced the highest mean AUC among the main full-data conditions.
- A mixed addition of 77 Normal and 99 Pneumonia images produced the highest mean balanced accuracy among the original synthetic conditions.
- Large-scale addition of 2,280 synthetic Normal images reduced performance.
- In the 25% real-data setting, synthetic augmentation did not outperform the real-only baseline.
- Repeating 176 real Normal images matched or exceeded the balanced-accuracy gain of 176 synthetic Normal images, suggesting that part of the gain may be explained by increased minority-class exposure.

**Submission scope.** This cleaned notebook documents the downstream pipeline and compact result summaries.

## 1. Reproducibility and execution notes

The notebook is designed for **Google Colab with a GPU runtime**.  
Previously generated images, checkpoints, and result CSV files are expected under:

`/content/drive/MyDrive/MedSymmFlow_Project`

The main results section loads saved experiment outputs and can be executed without rerunning all training jobs. The optional reproduction sections contain the core training and evaluation code.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q medmnist torchdiffeq diffusers accelerate zuko wandb     scikit-learn matplotlib tqdm opencv-python

In [ ]:
from pathlib import Path

REPO_DIR = Path("/content/MedSymmFlow")
if not REPO_DIR.exists():
    !git clone https://github.com/yovalyoval10-rgb/MedSymmFlow.git /content/MedSymmFlow

%cd /content/MedSymmFlow
# Pin the exact development revision used for the reported experiments.
!git checkout 908283d


In [ ]:
import copy
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from torchvision import transforms, models
from medmnist import PneumoniaMNIST

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PROJECT_DIR = Path("/content/drive/MyDrive/MedSymmFlow_Project")
RESULTS_DIR = PROJECT_DIR / "final_results"
SEEDS = [42, 123, 2026]

print("Device:", DEVICE)
print("Project directory exists:", PROJECT_DIR.exists())

## 2. Dataset and preprocessing

PneumoniaMNIST is a binary chest X-ray classification dataset:

- **Class 0:** Normal
- **Class 1:** Pneumonia
- **Training set:** 4,708 images
- **Validation set:** 524 images
- **Test set:** 624 images

The training set is imbalanced, with 1,214 Normal and 3,494 Pneumonia images. Images are resized to 224×224, converted to three channels, and normalized with ImageNet statistics to support ImageNet-pretrained ResNet-18.

In [ ]:
# Apply the same ImageNet-compatible preprocessing to every data source.
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

evaluation_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# Preserve the official validation and test splits for model selection and evaluation.
train_dataset = PneumoniaMNIST(
    split="train", transform=train_transform, download=True
)
val_dataset = PneumoniaMNIST(
    split="val", transform=evaluation_transform, download=True
)
test_dataset = PneumoniaMNIST(
    split="test", transform=evaluation_transform, download=True
)

def extract_labels(dataset):
    if hasattr(dataset, "labels"):
        return np.asarray(dataset.labels).reshape(-1).astype(int)
    if hasattr(dataset, "targets"):
        return np.asarray(dataset.targets).reshape(-1).astype(int)
    return np.asarray([
        int(np.asarray(dataset[i][1]).reshape(-1)[0])
        for i in range(len(dataset))
    ])

train_labels = extract_labels(train_dataset)

print("Train / validation / test:", len(train_dataset), len(val_dataset), len(test_dataset))
print("Training class counts:", dict(zip(*np.unique(train_labels, return_counts=True))))

## 3. Generative augmentation

MedSymmFlow was used to generate class-conditioned PneumoniaMNIST images. The project investigated several synthetic-data compositions:

| Condition | Synthetic addition |
|---|---:|
| Mixed small set | 77 Normal + 99 Pneumonia |
| Balanced moderate set | 300 per class |
| Normal-only small set | 176 Normal |
| Normal-only moderate set | 600 Normal |
| Full class balancing | 2,280 Normal |

Generated images were filtered using a pretrained classifier. For the Normal-only experiments, the images with the lowest predicted Pneumonia probability were retained.

In [ ]:
class SyntheticImageDataset(Dataset):
    """Loads PNG images from class folders named 'normal' and 'pneumonia'."""

    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []

        class_map = {"normal": 0, "pneumonia": 1}
        for class_name, label in class_map.items():
            class_dir = self.root_dir / class_name
            if not class_dir.exists():
                continue
            for image_path in sorted(class_dir.glob("*.png")):
                self.samples.append((image_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, np.asarray([label], dtype=np.int64)

## 4. Classifier and training protocol

A ResNet-18 pretrained on ImageNet was fine-tuned for binary classification.

**Fixed protocol**
- Optimizer: AdamW
- Learning rate: 1×10⁻⁴
- Weight decay: 1×10⁻⁴
- Batch size: 64
- Maximum epochs: 15
- Loss: class-weighted cross-entropy
- Model selection: highest validation AUC
- Random seeds: 42, 123, and 2026
- Primary decision threshold: 0.5
- Secondary analysis: threshold selected on the validation set to maximize balanced accuracy

In [ ]:
import copy
import random
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torchvision import models

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


# Select the decision threshold on validation predictions only.
def find_optimal_threshold(
    labels,
    probabilities,
    threshold_min=0.01,
    threshold_max=0.99,
    threshold_step=0.01
):
    labels = np.asarray(labels).astype(int)
    probabilities = np.asarray(probabilities)

    thresholds = np.arange(
        threshold_min,
        threshold_max + threshold_step,
        threshold_step
    )

    results = []

    for threshold in thresholds:
        predictions = (
            probabilities >= threshold
        ).astype(int)

        balanced_accuracy = balanced_accuracy_score(
            labels,
            predictions
        )

        results.append({
            "threshold": float(threshold),
            "balanced_accuracy": float(
                balanced_accuracy
            )
        })

    best_balanced_accuracy = max(
        row["balanced_accuracy"]
        for row in results
    )

    tied_results = [
        row
        for row in results
        if np.isclose(
            row["balanced_accuracy"],
            best_balanced_accuracy
        )
    ]

    best_result = min(
        tied_results,
        key=lambda row: abs(
            row["threshold"] - 0.5
        )
    )

    return (
        best_result["threshold"],
        best_result["balanced_accuracy"],
        results
    )


def calculate_threshold_metrics(
    labels,
    probabilities,
    threshold
):
    labels = np.asarray(labels).astype(int)
    probabilities = np.asarray(probabilities)

    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),
        "balanced_accuracy": balanced_accuracy_score(
            labels,
            predictions
        ),
        "f1": f1_score(
            labels,
            predictions
        ),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro"
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


# Seed every random source to make comparisons reproducible across conditions.
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def prepare_labels(labels):
    if isinstance(labels, np.ndarray):
        labels = torch.from_numpy(labels)

    return labels.long().view(-1)


def collect_predictions(
    model,
    loader,
    DEVICE
):
    model.eval()

    all_labels = []
    all_probabilities = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(
                DEVICE,
                non_blocking=True
            )

            labels = prepare_labels(labels).to(
                DEVICE,
                non_blocking=True
            )

            logits = model(images)

            probabilities = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            all_labels.extend(
                labels.detach().cpu().numpy()
            )

            all_probabilities.extend(
                probabilities.detach().cpu().numpy()
            )

    return (
        np.asarray(all_labels),
        np.asarray(all_probabilities)
    )


def train_single_run_with_threshold(
    seed,
    condition,
    train_dataset_for_run,
    class_weights,
    num_epochs=15
):
    set_seed(seed)

    generator = torch.Generator()
    generator.manual_seed(seed)

    current_train_loader = DataLoader(
        train_dataset_for_run,
        batch_size=64,
        shuffle=True,
        generator=generator,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    model = models.resnet18(
        weights=models.ResNet18_Weights.IMAGENET1K_V1
    )

    model.fc = nn.Linear(
        model.fc.in_features,
        2
    )

    model = model.to(DEVICE)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    # Retain the checkpoint with the highest validation AUC.
    best_val_auc = -np.inf
    best_epoch = None
    best_model_state = None

    for epoch in range(num_epochs):
        model.train()

        train_labels = []
        train_probabilities = []

        for images, labels in current_train_loader:
            images = images.to(
                DEVICE,
                non_blocking=True
            )

            labels = prepare_labels(labels).to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            probabilities = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            train_labels.extend(
                labels.detach().cpu().numpy()
            )

            train_probabilities.extend(
                probabilities.detach().cpu().numpy()
            )

        train_auc = roc_auc_score(
            train_labels,
            train_probabilities
        )

        val_labels, val_probabilities = collect_predictions(
            model,
            val_loader,
            DEVICE
        )

        val_auc = roc_auc_score(
            val_labels,
            val_probabilities
        )

        print(
            f"{condition} | seed {seed} | "
            f"epoch {epoch + 1:02d}/{num_epochs} | "
            f"train AUC {train_auc:.4f} | "
            f"val AUC {val_auc:.4f}"
        )

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch + 1
            best_model_state = copy.deepcopy(
                model.state_dict()
            )

    model.load_state_dict(
        best_model_state
    )

    val_labels, val_probabilities = collect_predictions(
        model,
        val_loader,
        DEVICE
    )

    (
        optimal_threshold,
        optimal_val_balanced_accuracy,
        _
    ) = find_optimal_threshold(
        val_labels,
        val_probabilities
    )

    test_labels, test_probabilities = collect_predictions(
        model,
        test_loader,
        DEVICE
    )

    test_auc = roc_auc_score(
        test_labels,
        test_probabilities
    )

    metrics_t05 = calculate_threshold_metrics(
        test_labels,
        test_probabilities,
        threshold=0.5
    )

    metrics_opt = calculate_threshold_metrics(
        test_labels,
        test_probabilities,
        threshold=optimal_threshold
    )

    result = {
        "condition": condition,
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_auc": best_val_auc,
        "optimal_threshold": optimal_threshold,
        "optimal_val_balanced_accuracy":
            optimal_val_balanced_accuracy,
        "test_auc": test_auc,

        "accuracy_t05": metrics_t05["accuracy"],
        "balanced_accuracy_t05":
            metrics_t05["balanced_accuracy"],
        "f1_t05": metrics_t05["f1"],
        "macro_f1_t05": metrics_t05["macro_f1"],
        "tn_t05": metrics_t05["tn"],
        "fp_t05": metrics_t05["fp"],
        "fn_t05": metrics_t05["fn"],
        "tp_t05": metrics_t05["tp"],

        "accuracy_opt": metrics_opt["accuracy"],
        "balanced_accuracy_opt":
            metrics_opt["balanced_accuracy"],
        "f1_opt": metrics_opt["f1"],
        "macro_f1_opt": metrics_opt["macro_f1"],
        "tn_opt": metrics_opt["tn"],
        "fp_opt": metrics_opt["fp"],
        "fn_opt": metrics_opt["fn"],
        "tp_opt": metrics_opt["tp"]
    }

    del model
    del optimizer
    del current_train_loader

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


print("Threshold-aware training function restored.")

## 5. Experiment design

### Full-data experiments
All 4,708 real training images were used, with or without synthetic additions.

### Low-data experiments
A stratified 25% subset of the real training set was selected separately for each seed. Three conditions were compared:
1. 25% real data only
2. 25% real + 44 synthetic Normal images
3. 25% real + 176 synthetic Normal images

### Real-image oversampling control
To test whether gains were specific to synthetic content, 176 real Normal training images were sampled and added a second time to the training dataset. Because they pass through the training transform again, repeated appearances may receive different stochastic preprocessing when such augmentations are enabled.

## 6. Load saved experiment results

This section loads the final CSV files saved during the experimental workflow. It avoids repeating expensive training runs during grading or presentation.

In [ ]:
FULL_RESULTS_PATH = PROJECT_DIR / "all_six_conditions_results.csv"
LOW_DATA_RESULTS_PATH = (
    PROJECT_DIR / "low_data_experiment"
    / "low_data_25_percent_all_conditions_results.csv"
)
REAL_CONTROL_RESULTS_PATH = (
    PROJECT_DIR / "real_oversampling_control"
    / "real_plus_176_repeated_real_normal_results.csv"
)

# Reuse saved experiment outputs instead of rerunning expensive training.
required_files = [
    FULL_RESULTS_PATH,
    LOW_DATA_RESULTS_PATH,
    REAL_CONTROL_RESULTS_PATH,
]

missing = [path for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing saved result files:\n" + "\n".join(map(str, missing))
    )

full_results = pd.read_csv(FULL_RESULTS_PATH)
low_data_results = pd.read_csv(LOW_DATA_RESULTS_PATH)
real_control_results = pd.read_csv(REAL_CONTROL_RESULTS_PATH)

print("Full-data rows:", len(full_results))
print("Low-data rows:", len(low_data_results))
print("Real-control rows:", len(real_control_results))

## 7. Full-data results

The primary metric is **balanced accuracy**, because the training data are class-imbalanced. Mean ± sample standard deviation are reported across three seeds.

In [ ]:
full_condition_names = {
    "real_only": "Real only",
    "real_plus_synthetic": "Real + 77 Normal + 99 Pneumonia",
    "real_plus_300_each": "Real + 300 per class",
    "real_plus_176_normal_only": "Real + 176 Synthetic Normal",
    "real_plus_600_normal_only": "Real + 600 Synthetic Normal",
    "real_plus_full_balance": "Real + 2280 Synthetic Normal",
}

full_summary = (
    full_results.assign(
        condition_label=full_results["condition"].map(full_condition_names)
    )
    .groupby("condition_label", sort=False)
    .agg(
        test_auc=("auc", ["mean", "std"]),
        accuracy=("accuracy", ["mean", "std"]),
        balanced_accuracy=("balanced_accuracy", ["mean", "std"]),
        macro_f1=("macro_f1", ["mean", "std"]),
    )
)

full_summary

### Interpretation

- The effect of synthetic augmentation was **non-monotonic**.
- The mixed set of 77 Normal and 99 Pneumonia images achieved the highest mean balanced accuracy among the original synthetic conditions.
- The 176-Normal condition achieved the highest mean AUC.
- Increasing the Normal synthetic set to 2,280 images reduced balanced accuracy below the real-only baseline.

## 8. Low-data results

In [ ]:
low_condition_names = {
    "real_25_percent_only": "25% Real only",
    "real_25_percent_plus_44_normal": "25% Real + 44 Synthetic Normal",
    "real_25_percent_plus_176_normal": "25% Real + 176 Synthetic Normal",
}

low_summary = (
    low_data_results.assign(
        condition_label=low_data_results["condition"].map(low_condition_names)
    )
    .groupby("condition_label", sort=False)
    .agg(
        test_auc=("test_auc", ["mean", "std"]),
        balanced_accuracy_at_05=("balanced_accuracy_t05", ["mean", "std"]),
        balanced_accuracy_selected=("balanced_accuracy_opt", ["mean", "std"]),
        selected_threshold=("optimal_threshold", ["mean", "std"]),
    )
)

low_summary

### Interpretation

Synthetic augmentation did not outperform the real-only baseline when only 25% of the real training data were used. Reducing the synthetic addition from 176 to 44 images reduced some of the degradation after threshold selection, but did not create a consistent improvement.

## 9. Matched real-image oversampling control

In [ ]:
# Compare synthetic augmentation with a matched repeated-real control.
control_summary = pd.DataFrame({
    "Condition": [
        "Real only",
        "Real + 176 Synthetic Normal",
        "Real + 176 Repeated Real Normal",
    ],
    "Test AUC": [
        "0.9716 ± 0.0023",
        "0.9772 ± 0.0037",
        "0.9724 ± 0.0064",
    ],
    "Accuracy": [
        "0.8798 ± 0.0192",
        "0.8953 ± 0.0158",
        "0.9065 ± 0.0301",
    ],
    "Balanced Accuracy": [
        "0.8409 ± 0.0261",
        "0.8624 ± 0.0217",
        "0.8779 ± 0.0421",
    ],
    "Macro F1": [
        "0.8609 ± 0.0247",
        "0.8807 ± 0.0195",
        "0.8940 ± 0.0365",
    ],
})

control_summary

In [ ]:
control_means = np.array([0.8409, 0.8624, 0.8779])
control_stds = np.array([0.0261, 0.0217, 0.0421])
control_labels = [
    "Real only",
    "Real + 176\nSynthetic Normal",
    "Real + 176\nRepeated Real Normal",
]

plt.figure(figsize=(9, 5.5))
bars = plt.bar(
    np.arange(3),
    control_means,
    yerr=control_stds,
    capsize=6,
)
plt.xticks(np.arange(3), control_labels)
plt.ylabel("Test Balanced Accuracy")
plt.title("Synthetic augmentation versus real-class oversampling")
plt.ylim(0.80, 0.94)
plt.grid(axis="y", alpha=0.3)

for bar, value in zip(bars, control_means):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.005,
        f"{value:.3f}",
        ha="center",
    )

plt.tight_layout()
plt.show()

### Interpretation

The repeated-real control achieved a higher mean balanced accuracy than the matched synthetic condition, although with substantially greater variability across seeds. This result indicates that part of the apparent augmentation benefit may come from additional exposure to the minority class rather than uniquely from new synthetic content. The synthetic condition nevertheless achieved the highest mean AUC, indicating that the two approaches may affect ranking and thresholded classification differently.

## 10. Image-quality and diversity analysis

Generated Normal images were compared with real Normal PneumoniaMNIST images.

| Analysis | Result |
|---|---:|
| FID, real vs. synthetic | 55.33 ± 0.61 |
| KID, real vs. synthetic | 0.06321 ± 0.00065 |
| Matched-size real vs. real FID | 24.18 ± 0.36 |
| Matched-size real vs. synthetic FID | 63.82 ± 0.74 |
| Synthetic-to-real nearest-neighbor cosine similarity | 0.9442 |
| Synthetic-to-synthetic nearest-neighbor similarity | 0.9669 |
| Real-to-real nearest-neighbor similarity | 0.9517 |

The synthetic images were visually related to the real distribution but remained measurably different. Their higher within-synthetic similarity suggests lower diversity than the real images. The nearest-neighbor analysis did not provide clear evidence of simple memorization.

## 11. Overall conclusions

1. MedSymmFlow-generated images can improve selected downstream metrics under some full-data augmentation settings.
2. The benefit is not monotonic with the number of generated images.
3. Small and moderate additions were generally more useful than large-scale class balancing.
4. Synthetic augmentation did not improve performance in the tested 25% real-data setting.
5. A matched repeated-real control showed that minority-class exposure and class balancing explain at least part of the observed gain.
6. Synthetic image quality, diversity, quantity, and class composition must therefore be evaluated jointly rather than assuming that more generated data will always improve classification.

## 12. Limitations

- Only three random seeds were used.
- PneumoniaMNIST is a small, low-resolution benchmark and may not represent full clinical chest X-ray complexity.
- The task is binary classification only.
- Synthetic images were filtered using a classifier, which may introduce selection bias.
- FID and KID are generic image-distribution metrics rather than clinical validity measures.
- No radiologist evaluation was performed.
- The threshold selected on the validation set was unstable in some conditions and did not always generalize to the test set.

## 13. Future work

- Repeat the most important comparisons with more random seeds.
- Compare synthetic augmentation with standard minority-class sampling and stronger conventional image augmentation.
- Evaluate generated images using clinically meaningful features or expert review.
- Test the approach on a higher-resolution chest X-ray dataset.
- Investigate diversity-aware selection of synthetic samples rather than confidence-only filtering.
- Explore training objectives that jointly optimize generation and class consistency.

## 14. Saved artifacts

The main project directory contains:

- `all_six_conditions_results.csv`
- `low_data_experiment/low_data_25_percent_all_conditions_results.csv`
- `real_oversampling_control/real_plus_176_repeated_real_normal_results.csv`
- `final_results/main_results_table.csv`
- `final_results/final_balanced_accuracy_all_conditions.png`
- `final_results/full_data_real_vs_synthetic_control.png`

These files allow the tables and plots to be regenerated without repeating all training runs.